## SSHOMP GUI curation tool

### Overview

The SSHOMP GUI Curation Tool is an interactive web application designed to facilitate the filtering, visualization, and export of SSHOMP dataset entries for curation. It provides a user-friendly interface that enables users to refine datasets using multiple criteria and export the selected entries as CSV files for further processing.

### Features

Dynamic Treemap Visualization: Displays the hierarchical structure of SSHOMP dataset entries, allowing users to explore different categories.

Filterable Dataset: Users can refine dataset entries using multiple filters including:

Activity count

Keywords count

Discipline count

Description length

Contributor category (User-Curated, Moderator, System-Generated)

URL Status (Available, Unavailable, Error, No URL Provided)

Inline Distribution Graphs: Log-scale line graphs provide a quick overview of the distribution of values for each numerical filter.

Click-based Selection: Selecting a source in the treemap refines the export to include only entries from that source.

CSV Export: Users can download the filtered dataset as a CSV file with a single click.

### How to Use

Get MP data: use local files or download recent files from SSHOMP. URLs can be checked, beware this 

Launch the Application: The tool runs in a Jupyter Notebook and can be started by executing the provided script.

Adjust Filters: Modify sliders and dropdowns to refine the dataset based on desired criteria.

Explore Treemap: Click on categories to drill down and select specific sources.

Export Data: Click the "Download CSV of Filtered Data" button to save the curated dataset for further analysis.

#Import libraries
import pandas as pd
import ast  # To convert string representations of dictionaries
import plotly.graph_objects as go
from dash import Dash, dcc, html, Input, Output
from IPython.display import clear_output
from sshmarketplacelib import MPData as mpd
from tqdm import tqdm #for the progress bar
import requests
import numpy as np

In [1]:
#download all categories of items from the MP
#True = uses local files where possible
mpdata = mpd()
df_tool_flat =mpdata.getMPItems ("toolsandservices", True)
df_publication_flat =mpdata.getMPItems ("publications", True)
df_trainingmaterials_flat =mpdata.getMPItems ("trainingmaterials", True)
df_workflows_flat =mpdata.getMPItems ("workflows", True)
df_datasets_flat =mpdata.getMPItems ("datasets", True)

NameError: name 'mpd' is not defined

In [3]:
#merge dfs
frames = [df_tool_flat, df_publication_flat, df_trainingmaterials_flat, df_workflows_flat, df_datasets_flat]
full_df = pd.concat(frames)
full_df.head()

,id,category,label,persistentId,lastInfoUpdate,status,description,contributors,properties,externalIds,...,thumbnail.concept.vocabulary.namespace,thumbnail.concept.vocabulary.label,thumbnail.concept.vocabulary.closed,thumbnail.concept.label,thumbnail.concept.notation,thumbnail.concept.uri,thumbnail.concept.candidate,dateCreated,dateLastUpdated,composedOf
0,72078,tool-or-service,140kit,SIU1nO,2025-01-30T09:59:57+0000,approved,140kit provides a management layer for tweet c...,"[{'actor': {'id': 2224, 'name': 'Ian Pearce', ...","[{'type': {'code': 'mode-of-use', 'label': 'Mo...",[],...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,72079,tool-or-service,360 stopni: 360-degree documentation,tZUwaR,2025-01-30T09:59:58+0000,approved,"Visit, without leaving home.\nA service involv...","[{'actor': {'id': 12862, 'name': 'University o...","[{'type': {'code': 'terms-of-use', 'label': 'T...",[],...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,72080,tool-or-service,3D fotogrametria: 3D models of any objects (ph...,tLdtav,2025-01-30T09:59:59+0000,approved,Everything can be scanned\nThe photogrammetric...,"[{'actor': {'id': 12860, 'name': 'Maria Curie-...","[{'type': {'code': 'terms-of-use', 'label': 'T...",[],...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,36324,tool-or-service,3DF Zephyr - photogrammetry software - 3d mode...,4gDAHv,2022-01-13T11:49:02+0000,approved,3DF Zephyr\[1\]\[2\] is a commercial photogram...,[],"[{'type': {'code': 'language', 'label': 'Langu...",[],...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,72082,tool-or-service,3DHOP: 3D Heritage Online Presenter,OgfTLA,2025-01-30T10:00:01+0000,approved,3DHOP (3D Heritage Online Presenter) is an op...,"[{'actor': {'id': 9747, 'name': 'Marco Callier...","[{'type': {'code': 'language', 'label': 'Langu...","[{'identifierService': {'code': 'GitHub', 'lab...",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [4]:
full_df['source.label'] = full_df['source.label'].fillna('user-created') #we need source labels for all items

In [56]:
def check_properties(id):  # Parses properties as a dict for easier inspection
    target_row = full_df.loc[full_df['persistentId'] == id]
    
    if target_row.empty:
        return {}  # Return an empty dict if no row matches the `persistentId`
    
    # Get the first matching row's `properties` value
    properties = target_row['properties'].iloc[0]  # Safe way to get a single item
    
    # Count the occurrences for each property code
    activity = sum(1 for i in properties if i.get('type', {}).get('code') == 'activity')
    keywords = sum(1 for i in properties if i.get('type', {}).get('code') == 'keyword')
    discipline = sum(1 for i in properties if i.get('type', {}).get('code') == 'discipline')
    language = sum(1 for i in properties if i.get('type', {}).get('code') == 'language')
    tool_family = sum(1 for i in properties if i.get('type', {}).get('code') == 'tool-family')
    modeofuse = sum(1 for i in properties if i.get('type', {}).get('code') == 'mode-of-use')
    intended_audience = sum(1 for i in properties if i.get('type', {}).get('code') == 'intended-audience')
    see_also = sum(1 for i in properties if i.get('type', {}).get('code') == 'see-also')
    user_manual_url = sum(1 for i in properties if i.get('type', {}).get('code') == 'user-manual-url')
    helpdesk_url = sum(1 for i in properties if i.get('type', {}).get('code') == 'helpdesk-url')
    license = sum(1 for i in properties if i.get('type', {}).get('code') == 'license')
    terms_of_use_url = sum(1 for i in properties if i.get('type', {}).get('code') == 'terms-of-use-url')
    technical_readiness_level = sum(1 for i in properties if i.get('type', {}).get('code') == 'technology-readiness-level')
    resource_category = sum(1 for i in properties if i.get('type', {}).get('code') == 'resource-category')
    version = sum(1 for i in properties if i == 'version')
    
    properties_dict = {
        "activity": activity, 
        "keywords": keywords, 
        "discipline": discipline, 
        "language": language,
        "tool-family": tool_family, 
        "mode of use": modeofuse, 
        "intended audience": intended_audience, 
        "see also": see_also, 
        "user manual URL": user_manual_url, 
        "helpdesk URL": helpdesk_url, 
        "license": license, 
        "terms of use URL": terms_of_use_url, 
        "technical readiness level": technical_readiness_level, 
        "resource category": resource_category, 
        "version": version
    }
    return properties_dict

# Apply the function row by row and assign the result to the new column
full_df['properties_dict'] = full_df['persistentId'].apply(check_properties)


In [60]:
full_df.head()

,id,category,label,persistentId,lastInfoUpdate,status,description,contributors,properties,externalIds,...,dateLastUpdated,composedOf,properties_dict,Contributor Category,description_length,url_status,activity,keywords,discipline,missing_external_ids
0,72078,tool-or-service,140kit,SIU1nO,2025-01-30T09:59:57+0000,approved,140kit provides a management layer for tweet c...,"[{'actor': {'id': 2224, 'name': 'Ian Pearce', ...","[{'type': {'code': 'mode-of-use', 'label': 'Mo...",[],...,NaN,NaN,"{'activity': 5, 'keywords': 3, 'discipline': 0...",User-Curated,571,No URL provided,0,0,0,True
1,72079,tool-or-service,360 stopni: 360-degree documentation,tZUwaR,2025-01-30T09:59:58+0000,approved,"Visit, without leaving home.\nA service involv...","[{'actor': {'id': 12862, 'name': 'University o...","[{'type': {'code': 'terms-of-use', 'label': 'T...",[],...,NaN,NaN,"{'activity': 11, 'keywords': 1, 'discipline': ...",User-Curated,428,No URL provided,0,0,0,True
2,72080,tool-or-service,3D fotogrametria: 3D models of any objects (ph...,tLdtav,2025-01-30T09:59:59+0000,approved,Everything can be scanned\nThe photogrammetric...,"[{'actor': {'id': 12860, 'name': 'Maria Curie-...","[{'type': {'code': 'terms-of-use', 'label': 'T...",[],...,NaN,NaN,"{'activity': 6, 'keywords': 1, 'discipline': 2...",User-Curated,900,No URL provided,0,0,0,True
3,36324,tool-or-service,3DF Zephyr - photogrammetry software - 3d mode...,4gDAHv,2022-01-13T11:49:02+0000,approved,3DF Zephyr\[1\]\[2\] is a commercial photogram...,[],"[{'type': {'code': 'language', 'label': 'Langu...",[],...,NaN,NaN,"{'activity': 0, 'keywords': 0, 'discipline': 0...",System-Generated,534,No URL provided,0,0,0,True
4,72082,tool-or-service,3DHOP: 3D Heritage Online Presenter,OgfTLA,2025-01-30T10:00:01+0000,approved,3DHOP (3D Heritage Online Presenter) is an op...,"[{'actor': {'id': 9747, 'name': 'Marco Callier...","[{'type': {'code': 'language', 'label': 'Langu...","[{'identifierService': {'code': 'GitHub', 'lab...",...,NaN,NaN,"{'activity': 0, 'keywords': 6, 'discipline': 2...",User-Curated,1141,No URL provided,0,0,0,False


In [7]:
# Lists for system accounts and moderators
system_accounts = ["Administrator", "Contributor", 'DACE Importer', 'Moderator', 'System importer', 'System moderator']
moderators = ["Michael Kurzmeier", "Laure Barbot", "Edward Gray", 'Alexander König', 'Irena Vipavc Brvar', 'Canan Arikan-Caba','Klaus Illmayer', 'Cesare Concordia', 'Clara Parente Boavida', 'Stefan Buddenbohm', 'Elena Battaner Moro', 'Christian Schuster', 'Magdalena Wnuk', 'Cristina Grisot', 'Barbara McGillivray', 'Maja Dolinar' ]

# Classify contributors
def classify_contributors(contributors_str):
    try:
        contributors = contributors_str  # Parse JSON-like string
        print(contributors)
        names = [contributor['actor']['name'] for contributor in contributors]  # Extract names
        
        # Identify account types
        has_system = any(name in system_accounts for name in names)
        has_moderator = any(name in moderators for name in names)
        has_user = any(name not in system_accounts + moderators for name in names)
        
        # Classification logic
        if has_user:
            return "User-Curated"
        elif has_system and not has_user and not has_moderator:
            return "System Account"
        elif has_moderator:
            return "Moderator"
        else:
            return 'System Account'
    except Exception as e:
        return "Unknown"

# Apply classification
full_df['Contributor Category'] = full_df['contributors'].apply(classify_contributors)


IOPub data rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_data_rate_limit`.

Current values:
ServerApp.iopub_data_rate_limit=1000000.0 (bytes/sec)
ServerApp.rate_limit_window=3.0 (secs)



In [8]:
full_df['Contributor Category'].value_counts()

Contributor Category
User-Curated      3489
System Account    1913
Moderator            3
Name: count, dtype: int64

In [10]:
# Add description length filter
def check_description(persID):
    target_rows = full_df.loc[full_df['persistentId'] == persID, 'description']
    if target_rows.empty:
        return 0
    description = target_rows.iloc[0]
    if description == 'No description provided.':
        return 0
    else:
        return len(description)

full_df['description_length'] = full_df['persistentId'].apply(check_description)

In [19]:
tqdm.pandas()
def check_accessibleAt(persId):
    # Fetch the target row
    target_row = full_df.loc[full_df['persistentId'] == persId]
    
    # Handle case where no matching row is found
    if target_row.empty:
        return {'error': 'Persistent ID not found'}
    
    # Extract the URL(s)
    url_list = target_row['accessibleAt'].values[0]  # Assuming it's a list of URLs or a single URL string

    # Handle empty URLs
    if not url_list or url_list == "":
        return {'No URL': 'No URL provided'}

    # Convert to a list if it's a single string
    if isinstance(url_list, str):
        url_list = [url_list]

    url_status = {}
    for url in url_list:
        try:
            status = requests.head(url, timeout=10).status_code
            if status == 200:
                url_status[url] = "available"
            else:
                url_status[url] = "unavailable"
        except requests.RequestException:
            url_status[url] = "error"

    return url_status

# Apply function with a progress bar
full_df['url_status'] = full_df['persistentId'].progress_apply(check_accessibleAt)

100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 5405/5405 [40:51<00:00,  2.20it/s]


In [32]:
full_df['url_status'] = {'https://github.com/WebEcologyProject/140kit': 'available'} #debug shortcut

In [20]:
full_df.to_csv('fulldata.csv')

In [35]:
df.to_csv('fulldata.csv')

to do

- [] add metadata into item box
- [x] add different filters
- [x] add export to csv 

In [ ]:
#switch to use full or sam[ple dataset

In [2]:
#Load data
file_path = 'fulldata.csv'
df = pd.read_csv(file_path)

In [57]:
df = full_df

In [3]:
df['source.label'] = df['source.label'].fillna('user-created') #we need source labels for all items

In [4]:
#Convert properties_dict and url_status to dictionaries and handle errors
def safe_literal_eval(x):
    try:
        return ast.literal_eval(x) if pd.notna(x) else {}
    except (ValueError, SyntaxError):
        return {}

df['properties_dict'] = df['properties_dict'].apply(safe_literal_eval)
df['url_status'] = df['url_status'].apply(safe_literal_eval)

In [5]:
#Extract properties into separate columns for filtering
df['activity'] = df['properties_dict'].apply(lambda x: x.get('activity', 0))
df['keywords'] = df['properties_dict'].apply(lambda x: x.get('keywords', 0))
df['discipline'] = df['properties_dict'].apply(lambda x: x.get('discipline', 0))

In [6]:
#Add contributor categories
def assign_contributor_category(row):
    if 'User-Curated' in row['Contributor Category']:
        return 'User-Curated'
    elif 'Moderator' in row['Contributor Category']:
        return 'Moderator-Curated'
    else:
        return 'System-Generated'

df['Contributor Category'] = df.apply(assign_contributor_category, axis=1)
df['Contributor Category'].value_counts()

Contributor Category
User-Curated         3489
System-Generated     1913
Moderator-Curated       3
Name: count, dtype: int64

In [7]:
#Extract URL status values from dictionaries
def get_url_status(x):
    if isinstance(x, dict) and len(x) > 0:
        return list(x.values())[0]  # Get the first status value (e.g., 'available', 'error')
    return 'No URL provided'

df['url_status'] = df['url_status'].apply(get_url_status)

In [8]:
#Add description length filter
def check_description(persID):
    target_rows = df.loc[df['persistentId'] == persID, 'description']
    if target_rows.empty:
        return 0
    description = target_rows.iloc[0]
    if description == 'No description provided.':
        return 0
    else:
        return len(description)

df['description_length'] = df['persistentId'].apply(check_description)

In [9]:
#Add external IDs check
def check_external_ids(persID):
    target_row = df.loc[df['persistentId'] == persID]
    if target_row.empty or 'externalIds' not in target_row.columns:
        return True  # Mark as missing if not available

    external_ids = target_row['externalIds'].values
    
    # Ensure external_ids is a single value
    if isinstance(external_ids, np.ndarray) and len(external_ids) > 0:
        external_ids = external_ids[0]
    
    # Handle empty lists properly
    if isinstance(external_ids, list) and len(external_ids) == 0:
        return True  # No external IDs (missing)
    
    # Convert from string to Python list if needed
    try:
        if isinstance(external_ids, str):
            external_ids_list = ast.literal_eval(external_ids)
        else:
            external_ids_list = external_ids
        return not bool(external_ids_list)  # Returns True if empty, False if not
    except (ValueError, SyntaxError):
        return True  # Parsing error, assume missing

df['missing_external_ids'] = df['persistentId'].apply(check_external_ids)

In [10]:
#Debug information and cleanup
print("Activity Min:", df['activity'].min(), "Max:", df['activity'].max())
print("Keywords Min:", df['keywords'].min(), "Max:", df['keywords'].max())
print("Discipline Min:", df['discipline'].min(), "Max:", df['discipline'].max())
print("Description Length Min:", df['description_length'].min(), "Max:", df['description_length'].max())
print("URL Status Values:", df['url_status'].unique())

df['url_status'].fillna('No URL provided', inplace=True)

Activity Min: 0 Max: 28
Keywords Min: 0 Max: 57
Discipline Min: 0 Max: 26
Description Length Min: 0 Max: 4064
URL Status Values: ['available' 'unavailable' 'error' 'No URL provided']


/tmp/ipykernel_1143933/2340224186.py:8: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['url_status'].fillna('No URL provided', inplace=True)


In [11]:
#Select relevant columns for treemap visualization
df_filtered = df[['persistentId', 'label', 'category', 'description', 'source.label', 'activity', 'keywords', 'discipline', 'description_length', 'url_status', 'missing_external_ids', 'Contributor Category']].copy()
df_filtered.columns = ['persistentId', 'label', 'category', 'details', 'source', 'activity', 'keywords', 'discipline', 'description_length', 'url_status', 'missing_external_ids', 'Contributor Category']

In [12]:
#Add category nodes and source nodes for the treemap
df_filtered['CategoryNode'] = df_filtered['source'] + "_" + df_filtered['category']
df_filtered = pd.concat([
    df_filtered,
    pd.DataFrame({'persistentId': ['root'], 'label': ['All Sources'], 'category': [''], 'details': [''], 'source': [''], 'activity': [0], 'keywords': [0], 'discipline': [0], 'description_length': [0], 'url_status': ['No URL provided'], 'missing_external_ids': [False], 'Contributor Category': ['System-Generated']})
], ignore_index=True)


# Ensure all top-level nodes point to 'root' for consistent hierarchy
top_level_sources = df_filtered['source'].unique()
for source in top_level_sources:
    if source not in ['', 'root'] and ((df_filtered['persistentId'] == source) & (df_filtered['source'] == '')).any():
        df_filtered.loc[df_filtered['persistentId'] == source, 'source'] = 'root'

In [13]:
#Add intermediate missing nodes dynamically
unique_sources = df_filtered['source'].unique()
source_rows = []
for source in unique_sources:
    if source not in ['root', ''] and not ((df_filtered['persistentId'] == source) & (df_filtered['source'] == 'root')).any():
        source_rows.append({'persistentId': source, 'label': source, 'source': 'root', 'category': '', 'details': '', 'activity': 0, 'keywords': 0, 'discipline': 0, 'description_length': 0, 'url_status': 'No URL provided', 'missing_external_ids': False, 'Contributor Category': 'System-Generated'})

df_filtered = pd.concat([df_filtered, pd.DataFrame(source_rows)], ignore_index=True)

In [14]:
#Ensure every node has a parent
all_ids = set(df_filtered['persistentId'])
all_parents = set(df_filtered['source'])
missing_parents = all_parents - all_ids
parent_rows = [{'persistentId': parent, 'label': parent, 'source': 'root', 'category': '', 'details': '',
                 'activity': 0, 'keywords': 0, 'discipline': 0, 'description_length': 0, 'url_status': 'No URL provided',
                 'missing_external_ids': False, 'Contributor Category': 'System-Generated'} for parent in missing_parents if parent not in ['', 'root']]
if parent_rows:
    df_filtered = pd.concat([df_filtered, pd.DataFrame(parent_rows)], ignore_index=True)

In [15]:
#App setup
try:
    clear_output(wait=True)  # Clear the current output in the notebook
except Exception as e:
    print(f"Error clearing output: {e}")

# Create a new Dash app only if it doesn't already exist
if 'app' not in globals() or not isinstance(app, Dash):
    app = Dash(__name__)
else:
    print("Dash app already initialized.")


In [16]:
#layout
import plotly.express as px

def create_distribution_figure(column_name):
    fig = px.line(df_filtered.groupby(column_name).size().reset_index(name='count'),
                  x=column_name, y='count', title=f"Distribution of {column_name}", height=100, log_y=True)
    fig.update_layout(margin=dict(l=20, r=20, t=20, b=20), xaxis_title=None, yaxis_title=None)
    fig.update_traces(mode='lines+markers')
    return fig
app.layout = html.Div([
    html.H1("Treemap with Filters and CSV Export"),

    html.Div([
        html.Label("Activity Count:"),
dcc.Graph(figure=create_distribution_figure('activity')), 
        dcc.Slider(id='activity-slider', min=df_filtered['activity'].min(), max=df_filtered['activity'].max(), step=1, value=df_filtered['activity'].max(),
                   marks={i: str(i) for i in range(df_filtered['activity'].min(), df_filtered['activity'].max() + 1, max(1, (df_filtered['activity'].max() - df_filtered['activity'].min()) // 5))},
                   tooltip={"always_visible": True}),

        html.Label("Keywords Count:"),
dcc.Graph(figure=create_distribution_figure('keywords')), 
        dcc.Slider(id='keywords-slider', min=df_filtered['keywords'].min(), max=df_filtered['keywords'].max(), step=1, value=df_filtered['keywords'].max(),
                   marks={i: str(i) for i in range(df_filtered['keywords'].min(), df_filtered['keywords'].max() + 1, max(1, (df_filtered['keywords'].max() - df_filtered['keywords'].min()) // 5))},
                   tooltip={"always_visible": True}),

        html.Label("Discipline Count:"),
dcc.Graph(figure=create_distribution_figure('discipline')), 
        dcc.Slider(id='discipline-slider', min=df_filtered['discipline'].min(), max=df_filtered['discipline'].max(), step=1, value=df_filtered['discipline'].max(),
                   marks={i: str(i) for i in range(df_filtered['discipline'].min(), df_filtered['discipline'].max() + 1, max(1, (df_filtered['discipline'].max() - df_filtered['discipline'].min()) // 5))},
                   tooltip={"always_visible": True}),

        html.Label("Description Length:"),
dcc.Graph(figure=create_distribution_figure('description_length')), 
        dcc.Slider(id='description-slider', min=df_filtered['description_length'].min(), max=df_filtered['description_length'].max(), step=1, value=df_filtered['description_length'].max(),
                   marks={i: str(i) for i in range(df_filtered['description_length'].min(), df_filtered['description_length'].max() + 1, max(1, (df_filtered['description_length'].max() - df_filtered['description_length'].min()) // 5))},
                   tooltip={"always_visible": True}),

        html.Label("Contributor Category:"),
        dcc.Dropdown(id='contributor-category-dropdown',
                     options=[{'label': 'All', 'value': 'all'}] + [{'label': cat, 'value': cat} for cat in df_filtered['Contributor Category'].unique()],
                     value='all',
                     multi=False,
                     clearable=False),
              html.Label("URL Status:"),
dcc.Dropdown(id='url-status-dropdown',
             options=[{'label': 'All', 'value': 'all'}] + [{'label': status, 'value': status} for status in df_filtered['url_status'].unique()],
             value='No URL provided',
             multi=False,
             clearable=True),
    ]),

    dcc.Graph(id='treemap'),

    html.Button("Download CSV of Filtered Data", id="download-btn", n_clicks=0),
    dcc.Download(id="download-data"),

    html.Div(id='debug-output', style={'marginTop': 20, 'color': 'red'})
])

In [17]:
#update
@app.callback(
    [Output('treemap', 'figure'), Output('debug-output', 'children')],
    [Input('activity-slider', 'value'),
     Input('keywords-slider', 'value'),
     Input('discipline-slider', 'value'),
     Input('description-slider', 'value'),
     Input('contributor-category-dropdown', 'value'),
     Input('url-status-dropdown', 'value')]
)
def update_treemap(activity, keywords, discipline, description_length, contributor_category, url_status):
    filtered_df = df_filtered[(df_filtered['activity'] <= activity) &
                              (df_filtered['keywords'] <= keywords) &
                              (df_filtered['discipline'] <= discipline) &
                              (df_filtered['description_length'] <= description_length) &
                              ((df_filtered['url_status'] == url_status) | (url_status == 'all'))]


    if contributor_category != 'all':
        filtered_df = filtered_df[filtered_df['Contributor Category'] == contributor_category]

    debug_info = f"Filtered DataFrame contains {len(filtered_df)} rows."

    if filtered_df.empty:
        fig = go.Figure()
        fig.add_annotation(text="No data matches the current filters.",
                           showarrow=False, xref="paper", yref="paper", x=0.5, y=0.5,
                           font=dict(size=20))
    else:
        fig = go.Figure(go.Treemap(
            ids=filtered_df['persistentId'],
            labels=filtered_df['label'],
            parents=filtered_df['source'],
            customdata=filtered_df[['details']].values,
            hovertemplate="<b>%{label}</b><br><br>Details: %{customdata[0]}<extra></extra>",
            marker=dict(line=dict(width=1, color="black"))
        ))

        fig.update_layout(title='Treemap of SSHOMP items per source')

    return fig, debug_info


In [18]:
#export
import dash
@app.callback(
    Output("download-data", "data"),
    Input("download-btn", "n_clicks"),
    [Input('activity-slider', 'value'),
     Input('keywords-slider', 'value'),
     Input('discipline-slider', 'value'),
     Input('description-slider', 'value'),
     Input('contributor-category-dropdown', 'value'),
     Input('treemap', 'clickData')],
    prevent_initial_call=True
)
def export_csv(n_clicks, activity, keywords, discipline, description_length, contributor_category, clickData):
    ctx = dash.callback_context
    if not ctx.triggered or ctx.triggered[0]['prop_id'].split('.')[0] != "download-btn":
        return None  # Only trigger on export button click
    
    filtered_df = df_filtered[(df_filtered['activity'] <= activity) &
                              (df_filtered['keywords'] <= keywords) &
                              (df_filtered['discipline'] <= discipline) &
                              (df_filtered['description_length'] <= description_length)]

    if contributor_category != 'all':
        filtered_df = filtered_df[filtered_df['Contributor Category'] == contributor_category]
    
    # Filter by selected source in the treemap
    if clickData and 'points' in clickData:
        selected_source = clickData['points'][0]['id']
        filtered_df = filtered_df[filtered_df['source'] == selected_source]
    
    return dcc.send_data_frame(filtered_df.to_csv, "filtered_data.csv")

In [19]:
#Run the app safely without errors using Dash
if __name__ == '__main__':
    app.run(debug=True, use_reloader=False, port=8050)